# Phase 2 — Predictions Table + AWS Bedrock Insights

Implements the second half of Phase 2 from the project blueprint:

1. **Predictions table** — train the tuned XGBoost model on all data available before the most recent completed fantasy week, generate projections for that week, and save them (with the context features that drove them) to a predictions table. In Databricks this would be a Delta table synced to Lakebase for the FastAPI layer; locally it is `data/predictions.parquet`.
2. **OpenAI integration (RAG flow)** — for each player: *retrieve* the model's projection + supporting stats from the predictions table, *augment* a prompt with them, and *generate* a qualitative insight using **GPT-5.6 Terra** (or other OpenAI models).

**Setup:**
1. Add your credentials to the `.env` file in this directory:
   ```
   OPENAI_API_KEY=your_api_key_here
   OPENAI_API_BASE=https://your-custom-endpoint.com/v1  # Optional: for custom models
   ```
2. Alternative: Store in Databricks secrets (scope: `openai-creds`, key: `api-key`)

**Note:** GPT-5.6 Terra requires special access or a custom endpoint. Standard OpenAI models (gpt-4, gpt-3.5-turbo) work out of the box.

**Runs without OpenAI too:** if the API key is missing or the call fails, the notebook falls back to a deterministic template-based insight generator so the full pipeline (predictions → insights → saved table) still executes end to end.

In [0]:
# Install dependencies
%pip install xgboost scikit-learn openai python-dotenv

## Build the Predictions Table

Same preprocessing and leakage guard as the model-eval notebooks. The model trains on everything strictly before the target week (all prior seasons + current season up to W−2), early-stops on week W−1, and projects week W — the exact production fold layout validated in `walk_forward_model.ipynb`.

In [0]:
import os
import pandas as pd
import numpy as np
from xgboost import XGBRegressor

# Load gold table from Unity Catalog Delta table
if 'gold_df' not in globals():
    gold_df = spark.table("fantasy_football.gold.player_weeks").toPandas()

gold_df = gold_df[gold_df['week'] <= 17]  # fantasy season only
TARGET = 'fantasy_points_ppr'

eval_meta = gold_df[['player_id', 'player_name', 'recent_team', 'position',
                     'season', 'week', 'opponent']].copy()

df = gold_df.copy()
identifier_cols = ['player_id', 'player_name', 'recent_team', 'opponent', 'starting_qb_id', 'gameday']
df = df.drop(columns=[c for c in identifier_cols if c in df.columns])

same_week_outcome_cols = [
    'pass_attempts', 'completions', 'passing_yards', 'passing_tds', 'interceptions',
    'rush_attempts', 'rushing_yards', 'rushing_tds',
    'targets', 'receptions', 'receiving_yards', 'receiving_tds',
    'player_opportunities', 'team_total_opportunities', 'opportunity_share',
    'hvt_carries', 'hvt_targets', 'total_hvts',
    'team_pass_attempts', 'target_share',
    'player_air_yards', 'team_air_yards', 'air_yards_share',
    'wopr', 'snap_share',
]
df = df.drop(columns=[c for c in same_week_outcome_cols if c in df.columns])
df['position'] = eval_meta['position']
df = pd.get_dummies(df.fillna(0), columns=['position'], prefix='pos')
feature_cols = [c for c in df.columns if c != TARGET]

# Project the most recent completed fantasy week
PREDICT_SEASON = int(eval_meta['season'].max())
in_season = eval_meta['season'] == PREDICT_SEASON
PREDICT_WEEK = int(eval_meta.loc[in_season, 'week'].max())

test_mask = in_season & (eval_meta['week'] == PREDICT_WEEK)
val_mask = in_season & (eval_meta['week'] == PREDICT_WEEK - 1)
train_mask = ~in_season | (eval_meta['week'] <= PREDICT_WEEK - 2)

# Hyperparameters selected by the leakage-safe tuning in walk_forward_model.ipynb
model = XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=4, min_child_weight=5,
    subsample=0.8, colsample_bytree=0.8,
    early_stopping_rounds=20, random_state=42, n_jobs=-1,
)
model.fit(
    df.loc[train_mask, feature_cols], df.loc[train_mask, TARGET],
    eval_set=[(df.loc[val_mask, feature_cols], df.loc[val_mask, TARGET])],
    verbose=False,
)
preds = np.clip(model.predict(df.loc[test_mask, feature_cols]), 0, None)

# Predictions table: projection + the context that drove it (for RAG retrieval)
context_cols = ['implied_total', 'team_spread', 'team_win_prob', 'is_home',
                'temp', 'wind', 'is_bad_weather', 'is_dome',
                'fantasy_points_3wk_avg', 'depth_chart_rank',
                'opp_def_ppg_allowed', 'prev_season_ppg']
predictions_df = eval_meta.loc[test_mask].reset_index(drop=True)
predictions_df['projected_ppr'] = preds.astype(float).round(1)
predictions_df['actual_ppr'] = df.loc[test_mask, TARGET].astype(float).round(1).values  # retrospective run: week already played
predictions_df = pd.concat(
    [predictions_df, df.loc[test_mask, context_cols].round(2).reset_index(drop=True)], axis=1
)
predictions_df = predictions_df.sort_values('projected_ppr', ascending=False).reset_index(drop=True)

# Convert predictions to Spark DataFrame for Delta table write
preds_spark = spark.createDataFrame(predictions_df)

# MERGE predictions into Delta table (history table, not snapshot)
# Partition by (season, week) - replaceWhere ensures we update only the target week
preds_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", f"season = {PREDICT_SEASON} AND week = {PREDICT_WEEK}") \
    .saveAsTable("fantasy_football.gold.predictions")

print(f"Projected {PREDICT_SEASON} week {PREDICT_WEEK}: {len(predictions_df)} players")
print(f"✓ Merged predictions into fantasy_football.gold.predictions")
display(predictions_df.head(10)[['player_name', 'recent_team', 'position', 'opponent',
                                 'projected_ppr', 'actual_ppr', 'implied_total']])

## RAG Flow: Retrieve → Augment → Generate

For each player the prompt is *augmented* with everything retrieved from the predictions table — projection, recent form, Vegas context, matchup, weather — so the LLM writes from **our data**, not its stale training knowledge. This is the pattern described in the project blueprint (the CeeDee Lamb example).

Insights are generated for the top 15 projected players to bound cost; in production this would run for the full slate.

In [0]:
# ============================================================================
# RAG: RETRIEVE stats -> AUGMENT prompt -> GENERATE insight via OpenAI API
# ============================================================================
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

OPENAI_MODEL = 'gpt-5.6-terra'  # Requires special access or custom endpoint
# Standard alternatives: 'gpt-4', 'gpt-4-turbo', 'gpt-3.5-turbo'

# Load OpenAI API key from .env file, Databricks secrets, or environment
try:
    OPENAI_API_KEY = dbutils.secrets.get(scope="openai-creds", key="api-key")
except Exception:
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
    if not OPENAI_API_KEY:
        print("Warning: OpenAI API key not found in .env, secrets, or environment.")
        print("Will fall back to template insights.")

# Optional: Custom API endpoint for GPT-5.6 Terra or other custom models
OPENAI_API_BASE = os.environ.get('OPENAI_API_BASE')  # Set in .env if needed

N_INSIGHTS = 15


def build_prompt(row):
    """AUGMENT step: inject retrieved model output + context into the prompt."""
    venue = 'at home' if row['is_home'] else 'on the road'
    weather = ('indoors (dome)' if row['is_dome']
               else f"{row['temp']:.0f}F, {row['wind']:.0f} mph wind"
               + (' — bad weather game' if row['is_bad_weather'] else ''))
    return f"""You are a fantasy football analyst. Using ONLY the data below, write a 2-3 sentence
insight for this player's upcoming game. Mention the projection, one supporting factor,
and one risk factor. Do not invent injuries or news not present in the data.

Player: {row['player_name']} ({row['position']}, {row['recent_team']})
Opponent: {row['opponent']} ({venue})
Model projection: {row['projected_ppr']} PPR points
Recent form (3-week avg): {row['fantasy_points_3wk_avg']} PPR points
Previous season average: {row['prev_season_ppg']} PPR points
Vegas implied team total: {row['implied_total']} | spread: {row['team_spread']:+.1f} | win prob: {row['team_win_prob']:.0%}
Opponent defense allows {row['opp_def_ppg_allowed']} PPR points/game to {row['position']}s
Depth chart rank: {int(row['depth_chart_rank'])}
Weather: {weather}"""


def template_insight(row):
    """Offline fallback so the pipeline runs end-to-end without AWS credentials."""
    lean = 'favorable' if row['team_spread'] > 0 else 'tough'
    parts = [
        f"{row['player_name']} projects for {row['projected_ppr']} PPR points against {row['opponent']}.",
        f"Vegas implies a {row['implied_total']:.1f}-point team total in a {lean} game script, "
        f"and he has averaged {row['fantasy_points_3wk_avg']:.1f} points over the last three weeks.",
    ]
    if row['is_bad_weather']:
        parts.append("Bad weather is a downside risk for this game.")
    elif row['opp_def_ppg_allowed'] and row['opp_def_ppg_allowed'] < 15:
        parts.append(f"Risk: {row['opponent']} has been stingy against {row['position']}s "
                     f"({row['opp_def_ppg_allowed']:.1f} PPR pts/game allowed).")
    return ' '.join(parts)


def make_llm():
    """Initialize OpenAI client. Returns (model_name, generate_fn) where 
    generate_fn: prompt -> text. Falls back to template insights if API key 
    is missing or connection fails."""
    if not OPENAI_API_KEY:
        print("No OpenAI API key — using offline template insights.")
        return 'offline_template', None
    
    try:
        from openai import OpenAI
        
        # Initialize client with optional custom base URL
        client_kwargs = {'api_key': OPENAI_API_KEY}
        if OPENAI_API_BASE:
            client_kwargs['base_url'] = OPENAI_API_BASE
            print(f"Using custom API endpoint: {OPENAI_API_BASE}")
        
        client = OpenAI(**client_kwargs)
        
        # Smoke test: verify API key works
        client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[{'role': 'user', 'content': 'hi'}],
            max_tokens=10,
        )

        def gen(prompt):
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=500,
                temperature=0.4,
            )
            return resp.choices[0].message.content.strip()

        return OPENAI_MODEL, gen
    except Exception as e:
        print(f"OpenAI API unavailable ({type(e).__name__}) — using offline template insights.")
        return 'offline_template', None


def generate_insight(gen_fn, row):
    return gen_fn(build_prompt(row)) if gen_fn else template_insight(row)


llm_name, gen_fn = make_llm()
print(f"Insight generator: {llm_name}")
top = predictions_df.head(N_INSIGHTS).copy()
top['insight'] = [generate_insight(gen_fn, row) for _, row in top.iterrows()]
top['insight_source'] = llm_name

# Final output: predictions + insights table (Delta table in Databricks)
final = predictions_df.merge(
    top[['player_id', 'insight', 'insight_source']], on='player_id', how='left'
)
# Convert to Spark DataFrame and MERGE into predictions table
# This replaces the earlier predictions-only write with predictions + insights
final_spark = spark.createDataFrame(final)

# MERGE the predictions WITH insights using replaceWhere on (season, week)
# Enable schema evolution to add insight columns to existing table
final_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", f"season = {PREDICT_SEASON} AND week = {PREDICT_WEEK}") \
    .option("mergeSchema", "true") \
    .saveAsTable("fantasy_football.gold.predictions")

print(f"✓ Saved {PREDICT_SEASON} week {PREDICT_WEEK} predictions + insights to fantasy_football.gold.predictions\n")

for _, row in top.head(5).iterrows():
    print(f"--- {row['player_name']} ({row['position']}, {row['recent_team']}) "
          f"proj {row['projected_ppr']} | actual {row['actual_ppr']} ---")
    print(row['insight'], '\n')